In [0]:
import delta

def table_exists(catalog,database,table):
    count = (spark.sql(f"SHOW TABLES FROM {catalog}.{database}")
                .filter(f"database = '{database}' AND tableName = '{table}'")
                .count())
    return count == 1

In [0]:
catalog = "bronze"
schema = "upsell"
tablename = dbutils.widgets.get("tablename")
id_field = dbutils.widgets.get("id_field")
timestamp_field = dbutils.widgets.get("timestamp_field")

In [0]:
if not table_exists(catalog,schema,tablename):
    print('Tabela não existente. Criando...')
    df_full = spark.read.format("parquet").load(f"/Volumes/raw/upsell/full_load/{tablename}/")
    (df_full.coalesce(1)
        .write
        .format("delta")
        .saveAsTable(f"{catalog}.{schema}.{tablename}"))
else:
    print('Tabela já existente. Ignorando full-load.')

In [0]:
(spark.read
    .format("parquet")
    .load(f"/Volumes/raw/upsell/cdc/{tablename}/")
    .createOrReplaceTempView(f"view_{tablename}"))
    
query = f'''
select * from view_{tablename}
qualify ROW_NUMBER() OVER (partition by {id_field} order by {timestamp_field} desc) = 1
'''
df_cdc_unique = spark.sql(query)

In [0]:
bronze = delta.DeltaTable.forName(spark,f"{catalog}.{schema}.{tablename}")

#UPSERT 
(bronze.alias("b") 
    .merge(df_cdc_unique.alias("d"), f"b.{id_field} = d.{id_field}")
    .whenMatchedDelete(condition = "d.OP = 'DELETE'")
    .whenMatchedUpdateAll(condition = "d.OP ='UPDATE'") 
    .whenNotMatchedInsertAll(condition = "d.OP = 'INSERT' or d.OP = 'UPDATE' ") 
    .execute() 
)